# Sessionize page views with a 30-minute gap

**Track:** pyspark | **Difficulty:** medium | **Tags:** sessionization, window functions, lag, gaps and islands

Run the three setup cells below once, in order. Then read the problem, write your
answer in the answer cell so that it ends with a DataFrame called `result`, and run
`check(result)`. You can run the check as many times as you like.

In [ ]:
# Setup: installs PySpark on Colab and starts a local Spark session (about a minute).
import os
import re
import shutil
import subprocess
import sys

PYSPARK_VERSION = "4.1.1"

if "google.colab" in sys.modules:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", f"pyspark=={PYSPARK_VERSION}"],
        check=True,
    )
    # PySpark 4 needs Java 17 or newer.
    java_major = 0
    if shutil.which("java"):
        version_text = subprocess.run(["java", "-version"], capture_output=True, text=True).stderr
        match = re.search(r'version "(\d+)', version_text)
        java_major = int(match.group(1)) if match else 0
    if java_major < 17:
        subprocess.run(["apt-get", "install", "-y", "-qq", "openjdk-17-jdk-headless"], check=True)
        os.environ["JAVA_HOME"] = "/usr/lib/jvm/java-17-openjdk-amd64"

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder.master("local[2]")
    .appName("007-sessionize-page-views")
    .config("spark.sql.session.timeZone", "UTC")
    .config("spark.sql.shuffle.partitions", "4")
    .config("spark.ui.enabled", "false")
    .getOrCreate()
)
spark.sparkContext.setLogLevel("ERROR")
print("Spark", spark.version, "is ready.")

In [ ]:
#@title Load the data (run this cell, no need to read it)
import base64
import pathlib

_DATA = {
    "page_views": """
UEFSMRUEFeADFYoCTBU8FQASAADwAQQdAAkBABQJBwQACg0IAAYNCAARDQgABw0IABYNCAAbDQgAGg0IABkNCAAODQgAFw0IAAsN
CAAcDQgAFQ0IAAQNCAADDQgABQ0IAAwNCAAeDQgACQ0IABINCAAIDQgADQ0IAAINCAAPDQgAEA0IABMNCDwYAAAAAAAAAAEAAAAA
AAAAFQAVqgIVsgIsFbYDFRAVBhUGHBgIHgAAAAAAAAAYCAEAAAAAAAAAFgAoCB4AAAAAAAAAGAgBAAAAAAAAABERAAAAlQHwlAMA
AAC2AwEFOQAEMQgo5qCk1AqnsHNSQ+UMBwIA65EGXwwyAdhIaMRM+uIQojVQazENoEeObRdK20AbjQhCSmrCNpylOeM1eoJtL+md
03jvCWMKCEO/NMYNYolwStimXYMGGMK8YIrdA59TAwG8hpeaeHBYKhsdKqFjWDwlGHtaCPckvi0D3QC5UmmHnG2HkadeAAAAFQQV
wBoV9BZMFagDFQASAACgDWzAeuPa+C8GAECgBvf6LwYAABdxKwcwBgDAsJnEARBQgJUbpwAwBgBAt66TAzAGAABBxp//AShIvH/B
+S8GAIB5GdT7LwYAwAGu4wE4EACSw1oFASgIg528ARgMAFm8ygVgCBQ4xwEITIDXH/X9LwYAgDljQgYwBgBAlgqiAWAMAEgGCQFw
DACkcjIBWAxAZA6kARAMQLPmgAEgDIBVV+4FgAjsNkABSAzALwoWBWho6NHM9y8GAAAyXpoIMAYAwERp4AwwBgCANGsUAagMwFs8
gAGIDECJQOcBMBDA0f4K/AEwCMhUuQFIDIDvay8BYAxAnFVIATAsACbZMP4vBgDALkoqAagsAP560vYvBgAALzolASAMQCD7RAVw
CKlUkwHADADr5xoBqAwAj7FfBVgIxFsiAXAMANDEPQEoDMCGddQBGAxAQuHLAVgMAPgH5QFYDMB+NtQBgAwAe0jvARAMAEhXLgEo
DAB86DkFKAjj+tYFsAhtL+0BSAwAuLeuBYgIqXOdBbAIr81IAWgMgGbblAGADMAq7vMBQAzAli2WBagIZgL+BQgoAWIRATAGAEC+
YxYBCAwAAN9bAXgMQCyo9QFADMBmO78BEAwAsWXLBRAo1U2xAjAGAMAUe1wBQAzAdXwUIXgMQE29rSVgCHNFvAUgCAQqjQU4CAIS
YQEoDACQLbYBQCwAezcMCzAGAIA91UUBqAyAFdFJARgMQKSqFQHoLMAftfMEMAYAAIb78gE4DEDT82QBYAwAIYyKASgMAAfhsCEA
DIC9EXwBCAwAXEuFIQAMgC/6eQUgCEl+DAFYDEAe+loBgAyASLnNAfgMQJmr1gVQCCP/GEFADADcy1AFQAiXV1IBICzARUwpCDAG
AMDT160BQAzAwF4WBVgItbNxARAMACm++iFwDEDV1UQBKAyAYYqbAUAMQBxh+gHwDIA1FJwBYAyApHEyATgMwIXDDwFYDIAD8FYB
GAwAOu0oIWAMQGdRIwE4DAC9tMYB6AzAe0URBVgI3s6vASgMwB/mdwUICOt8swFIDIBSYKoF2AhBGxgFCAgGww8FCAiMZaMF4Ai4
VKYBMAwAaI3wQXAsQB4SVg0wBgBAr2+TARgMQKVpagFIDAD7cqsFOAzsTjsNASAI+L0eAXgMwOPSbQGQDMA2YeMBQAzAtBnvBSgI
JhjcAUAMwJaCDgEQDMCu8jwBKAyAe/bJJUAIdFgUBYAEj0BlECwAqewJ/S8GAMCeDUpBcAxA8noeASgMwI/6vAE4DECkYi0BSAwA
G1NeASgMQJvIDgGIDICsbTQhYAzAe05OAZAMgI7j4wVYCMNcIgUoCOOP2AXACM9N0QFIDMBhot0BYAwAMPl5AWAMQGllHkFQDADX
op4hcCwAxMzoBDAGAAAWvs8BSAzAzXY6IbAMAK3K6QEwDACpT6kB2AwArCL5AfAMAHNWwSHADMARQRoFQAgtHoQBIAxA3N8/JbAI
GfFJAXgMAJsoogUYCOet+kGQDMD4MKEBYAyAyTKvAQgMAFmY1gWoCHOmTwEIDEBaCqkB4AxA+xZrAWAQAPpVoQUBmCgl6MD7LwYA
QFr76gU4CBV5fgGgDIBPT0YlCAgtyoABwAyALePCBTgIMkJMAfAMgB1lwgEIDAB2vJ0FyAgMotlh8AxA2qP9BRAEGOwFUAzALQDv
AWgMQAh+hgEwLIAKuOoKMAYAwNucqgEQDEA3HE4B6AxAt98XAdgMQJu1/QEIDICeGfwBCAwAxC7xAVAMgM9a+AHoDIBkc7gB+AwA
aKO9BaAI59krARgMwOdEcQHADEBGc3ABGBCA4CaH/AHYCKDMNQEQDACOoeUBOAzA0sczAdAMAI2MgQUgCBD8MQEgDMDXco8B4AzA
vSCBAZgMADTBjAEgDIBBqugFCAjqXsuhqAwAFlfdBRAIXZ8rISg8wB33BwEwBgBAgZ5H/y8GABUAFdIDFdoDLBW2AxUQFQYVBhwY
CEBaCqkNMAYAGAgAuLeu9i8GABYAKAhAWgqpDTAGABgIALi3rvYvBgAREQAAAOkB8OgDAAAAtgMBCDkAAQIDBAUGBwgJCgsMDQ4P
EBESExQVFhcYGRobHB0eHyAhIiMkJSYnKCkqKywtLi8wMTIzNDU2Nzg5Ojs8PT4/QEFCQ0RFRkdINUlKS0xNTk9QUVJTVFVWV1hZ
WlsRXF1eX2BhYmNkZWZnaGlqa2xtbm9wcXJzdHV2d3h5ent8fX5/gIGCg4SFhoeIiYqLjI2Oj5CRkpOUlZaXmJmam5ydnp+goaKj
pKWmp6ipqqusrT6ur7CxsrO0tba3uLm6u7y9vr/AwcIRw8TFxknHyBLJysvMzc7P0BHR0tMAAAAAABUEFZQBFYgBTBUOFQASAABK
JAUAAAAvaG9tZQcBCRRvcmRlcnMJFAhlbHAFFBhzZWFyY2gJAR8gY2hlY2tvdXQIAQ08cHJvZHVjdAUAAAAvY2FydBUAFboBFcAB
LBW2AxUQFQYVBhw2ACgHL3NlYXJjaBgFL2NhcnQREQAAAF3wXAMAAAC2AwEDOYi0RVRZeurWLlrQygsyVF0pzI4hpC0ZCwvNEJxW
diK2kkmwFASU2e2IzWwJ1ItGC0YoLevS1GI9cME6AZ2xkIy9guhWk/MGbSTGokPICq3JIq0AABUEGUw1ABgGc2NoZW1hFQYAFQQl
AhgHdXNlcl9pZAAVBCUCGAl2aWV3ZWRfYXQlFEyMEhwsAAAAAAAVDCUCGARwYWdlJQBMHAAAABa2AxkcGTwmABwVBBk1AAYQGRgH
dXNlcl9pZBUCFrYDFq4HFuAFJrICJggcGAgeAAAAAAAAABgIAQAAAAAAAAAWACgIHgAAAAAAAAAYCAEAAAAAAAAAEREAGSwVBBUA
FQIAFQAVEBUCADwpBhkmALYDAAAAJgAcFQQZNQAGEBkYCXZpZXdlZF9hdBUCFrYDFrgfFvQbJv4cJugFHBgIQFoKqQ0wBgAYCAC4
t672LwYAFgAoCEBaCqkNMAYAGAgAuLeu9i8GABERABksFQQVABUCABUAFRAVAgA8KQYZJgC2AwAAACYAHBUMGTUABhAZGARwYWdl
FQIWtgMWwgMWvAMmhCMm3CEcNgAoBy9zZWFyY2gYBS9jYXJ0EREAGSwVBBUAFQIAFQAVEBUCADwWpBcZBhkmALYDAAAAFqgqFrYD
JggWkCUAGRwYDEFSUk9XOnNjaGVtYRjAAi8vLy8vK2dBQUFBUUFBQUFBQUFLQUF3QUJnQUZBQWdBQ2dBQUFBQUJCQUFNQUFBQUNB
QUlBQUFBQkFBSUFBQUFCQUFBQUFNQUFBQjhBQUFBTkFBQUFBUUFBQUNnLy8vL0FBQUJCUkFBQUFBY0FBQUFCQUFBQUFBQUFBQUVB
QUFBY0dGblpRQUFBQUFFQUFRQUJBQUFBTXovLy84QUFBRUtFQUFBQUNBQUFBQUVBQUFBQUFBQUFBa0FBQUIyYVdWM1pXUmZZWFFB
QmdBSUFBWUFCZ0FBQUFBQUFnQVFBQlFBQ0FBR0FBY0FEQUFBQUJBQUVBQUFBQUFBQVFJUUFBQUFJQUFBQUFRQUFBQUFBQUFBQndB
QUFIVnpaWEpmYVdRQUNBQU1BQWdBQndBSUFBQUFBQUFBQVVBQUFBQUFBQUFBABggcGFycXVldC1jcHAtYXJyb3cgdmVyc2lvbiAy
NS4wLjEZPBwAABwAABwAAAATAwAAUEFSMQ==
""",
}

_data_dir = pathlib.Path("data") / "007-sessionize-page-views"
_data_dir.mkdir(parents=True, exist_ok=True)
for _name, _encoded in _DATA.items():
    (_data_dir / f"{_name}.parquet").write_bytes(base64.b64decode(_encoded))

page_views = spark.read.parquet((_data_dir / "page_views.parquet").resolve().as_posix())
page_views.createOrReplaceTempView("page_views")

for _name in _DATA:
    print(f"{_name}: {spark.table(_name).count()} rows")

In [ ]:
#@title Grader (run this cell, no need to read it)
"""Result grading shared by the build script and every generated notebook.

build_problem.py pastes this file into the notebook verbatim, so it must stay
standalone: standard library only, no imports from the rest of the toolkit.

A result is reduced to a "summary": its columns and types, its row count and
SHA-256 fingerprints of its normalized rows. Only the summary of the correct
answer is shipped in a notebook, never the rows themselves.
"""
import datetime
import decimal
import hashlib
import json
import math

FLOAT_DECIMALS = 6


def normalize_value(value):
    """Turn one cell value into something json.dumps renders the same way everywhere."""
    if value is None or isinstance(value, (bool, int, str)):
        return value
    if isinstance(value, float):
        if math.isnan(value):
            return "NaN"
        if math.isinf(value):
            return "Infinity" if value > 0 else "-Infinity"
        return round(value, FLOAT_DECIMALS) + 0.0  # + 0.0 turns -0.0 into 0.0
    if isinstance(value, decimal.Decimal):
        return format(value, "f")
    if isinstance(value, (datetime.datetime, datetime.date)):
        return value.isoformat()
    if isinstance(value, (bytes, bytearray)):
        return bytes(value).hex()
    if hasattr(value, "asDict"):  # pyspark Row (struct column)
        return {key: normalize_value(item) for key, item in value.asDict().items()}
    if isinstance(value, dict):  # map column
        pairs = [[normalize_value(key), normalize_value(item)] for key, item in value.items()]
        return sorted(pairs, key=_encode)
    if isinstance(value, (list, tuple)):
        return [normalize_value(item) for item in value]
    return str(value)


def _encode(value):
    return json.dumps(value, sort_keys=True, separators=(",", ":"))


def _digest(lines):
    sha = hashlib.sha256()
    for line in lines:
        sha.update(line.encode("utf-8"))
        sha.update(b"\n")
    return sha.hexdigest()


def describe_schema(df):
    """Column names and Spark types, sorted by name so column order is ignored."""
    return sorted([field.name, field.dataType.simpleString()] for field in df.schema.fields)


def summarize_rows(columns, rows, order_matters=False):
    """columns: sorted [name, type] pairs. rows: normalized values in that column order."""
    encoded = [_encode(row) for row in rows]
    if not order_matters:
        encoded.sort()
    column_fingerprints = {}
    for index, (name, _) in enumerate(columns):
        column_fingerprints[name] = _digest(sorted(_encode(row[index]) for row in rows))
    return {
        "columns": [list(column) for column in columns],
        "row_count": len(rows),
        "order_matters": order_matters,
        "fingerprint": _digest([_encode(columns)] + encoded),
        "column_fingerprints": column_fingerprints,
    }


def summarize(df, order_matters=False):
    columns = describe_schema(df)
    names = [name for name, _ in columns]
    # collect() returns `timestamp` values in the machine's local time zone, which differs
    # between the build machine and Colab. Casting to string inside Spark uses the session
    # time zone instead (UTC in both places). `timestamp_ntz` columns are not affected.
    expressions = [
        "CAST(`%s` AS STRING) AS `%s`" % (name, name) if kind == "timestamp" else "`%s`" % name
        for name, kind in columns
    ]
    collected = df.selectExpr(*expressions).collect()
    rows = [[normalize_value(row[name]) for name in names] for row in collected]
    return summarize_rows(columns, rows, order_matters)


def grade(df, expected):
    """Compare a DataFrame to the expected summary. Returns (passed, hints).

    The hints say what kind of thing is wrong (columns, types, row count, which
    column) but never what the right values are.
    """
    if not hasattr(df, "schema") or not hasattr(df, "collect"):
        return False, ["check() needs a Spark DataFrame, got %s." % type(df).__name__]

    names = [field.name for field in df.schema.fields]
    duplicates = sorted({name for name in names if names.count(name) > 1})
    if duplicates:
        return False, ["Your result has duplicate column names: %s." % ", ".join(duplicates)]

    actual = dict(describe_schema(df))
    wanted = dict(expected["columns"])
    hints = []
    missing = sorted(set(wanted) - set(actual))
    extra = sorted(set(actual) - set(wanted))
    if missing:
        hints.append("Missing columns: %s." % ", ".join(missing))
    if extra:
        hints.append("Unexpected columns: %s." % ", ".join(extra))
    for name in sorted(set(wanted) & set(actual)):
        if wanted[name] != actual[name]:
            hints.append("Column %s has type %s, expected %s." % (name, actual[name], wanted[name]))
    if hints:
        return False, hints

    row_count = df.count()
    if row_count != expected["row_count"]:
        return False, [
            "Row count is wrong: your result has %d rows, expected %d."
            % (row_count, expected["row_count"])
        ]

    summary = summarize(df, expected["order_matters"])
    if summary["fingerprint"] == expected["fingerprint"]:
        return True, []

    wrong = sorted(
        name
        for name, fingerprint in summary["column_fingerprints"].items()
        if fingerprint != expected["column_fingerprints"][name]
    )
    if wrong:
        return False, ["Columns and row count are right, but values are wrong in: %s." % ", ".join(wrong)]
    if expected["order_matters"]:
        return False, ["The values are right, but the rows are in the wrong order."]
    return False, ["Each column has the right set of values, but they are combined into the wrong rows."]


def make_check(expected):
    def check(df):
        passed, hints = grade(df, expected)
        if passed:
            print("Correct. Your result matches the expected output.")
        else:
            print("Not correct yet.")
            for hint in hints:
                print("  - " + hint)
        return passed

    return check

EXPECTED = {'column_fingerprints': {'page_views': '63b9027a391f2ec72e38616d0fd93abce4a8a44a8b0d49385b80019de9bb161d',
                         'session_end': '664164e46951f53fd92d28b09131ae90c5cb0376840d5703c29170e12b55774f',
                         'session_number': 'fa741b85e32d070c90af3270bdcd4fb2709360489b61ca1018f8537657c70c87',
                         'session_start': '78280011556ba549f52748f6290eab98ab1f797d0878cff6b50adb02ae20b511',
                         'user_id': '604fe1e41f45d41b7093a1deb69e9aa9ea1ccd84073efa71398361854918109a'},
 'columns': [['page_views', 'bigint'],
             ['session_end', 'timestamp_ntz'],
             ['session_number', 'bigint'],
             ['session_start', 'timestamp_ntz'],
             ['user_id', 'bigint']],
 'fingerprint': 'bce8fd9a99c70427631c62459f5bb5bb0a586a8808833c6dd71a24ebf5f1c6b5',
 'order_matters': False,
 'row_count': 55}

check = make_check(EXPECTED)

## The situation

The product team wants session-level numbers: how many sessions a user has and how
long they last. The tracking system only records individual page views, so the
sessions have to be derived.

## Input tables

### `page_views`

One row per page view.

| Column | Type | Description |
|---|---|---|
| `user_id` | BIGINT | The user. |
| `viewed_at` | TIMESTAMP_NTZ | When the page was viewed. Two views of a user can share the same instant. |
| `page` | STRING | The page path. |

## Requirement

Group each user's page views into sessions and return one row per session.

- A user's first view starts their first session.
- A view starts a new session if it comes **more than 30 minutes** after the user's
  previous view. A gap of exactly 30 minutes continues the session.
- The gap is always measured from the previous view, not from the start of the
  session. A session can last for hours as long as no gap exceeds 30 minutes.
- Sessions do not stop at midnight.
- `session_number` counts a user's sessions in time order, starting at 1.

## Expected output

One row per session.

| Column | Type | Description |
|---|---|---|
| `user_id` | BIGINT | |
| `session_number` | BIGINT | 1 for the user's first session, 2 for the next, and so on. |
| `session_start` | TIMESTAMP_NTZ | Time of the first view in the session. |
| `session_end` | TIMESTAMP_NTZ | Time of the last view in the session. |
| `page_views` | BIGINT | Number of views in the session. |

In [ ]:
page_views.show(5)

## Your answer

Use the DataFrame API. The tables are available as DataFrames: `page_views`.

In [ ]:
result = None  # Replace None with your DataFrame.

In [ ]:
check(result)